In [19]:
import kagglehub
import os
import torch.nn as nn
import torch.functional as F
import torch.optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Now do your work and show us your results
Please use a neural network to solve this problem


Let's keep a +95% accuracy for now (this is changeable later)

In [3]:

# Download latest version
path = kagglehub.dataset_download("sahilislam007/college-student-placement-factors-dataset")

file_path = os.path.join(path, 'college_student_placement_dataset.csv')

import pandas as pd

df = pd.read_csv(file_path)

# Display the first 5 rows
display(df.head())

# Print column names and their data types
print(df.info())

# Display a concise summary of the DataFrame
display(df.describe())

,College_ID,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Internship_Experience,Extra_Curricular_Score,Communication_Skills,Projects_Completed,Placement
0,CLG0030,107,6.61,6.28,8,No,8,8,4,No
1,CLG0061,97,5.52,5.37,8,No,7,8,0,No
2,CLG0036,109,5.36,5.83,9,No,3,1,1,No
3,CLG0055,122,5.47,5.75,6,Yes,1,6,1,No
4,CLG0004,96,7.91,7.69,7,No,8,10,2,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   College_ID              10000 non-null  object 
 1   IQ                      10000 non-null  int64  
 2   Prev_Sem_Result         10000 non-null  float64
 3   CGPA                    10000 non-null  float64
 4   Academic_Performance    10000 non-null  int64  
 5   Internship_Experience   10000 non-null  object 
 6   Extra_Curricular_Score  10000 non-null  int64  
 7   Communication_Skills    10000 non-null  int64  
 8   Projects_Completed      10000 non-null  int64  
 9   Placement               10000 non-null  object 
dtypes: float64(2), int64(5), object(3)
memory usage: 781.4+ KB
None


,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Extra_Curricular_Score,Communication_Skills,Projects_Completed
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,99.471800,7.535673,7.532379,5.546400,4.970900,5.561800,2.513400
std,15.053101,1.447519,1.470141,2.873477,3.160103,2.900866,1.715959
min,41.000000,5.000000,4.540000,1.000000,0.000000,1.000000,0.000000
25%,89.000000,6.290000,6.290000,3.000000,2.000000,3.000000,1.000000
50%,99.000000,7.560000,7.550000,6.000000,5.000000,6.000000,3.000000
75%,110.000000,8.790000,8.770000,8.000000,8.000000,8.000000,4.000000
max,158.000000,10.000000,10.460000,10.000000,10.000000,10.000000,5.000000


In [7]:
# Target variable
y = df["Placement"]
if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)

X = df.drop("Placement", axis=1)


In [10]:

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ===============================
# Preprocessing (Sklearn pipeline)
# ===============================
numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [11]:
# Convert to tensors
X_train_t = torch.tensor(X_train.toarray() if hasattr(X_train, "toarray") else X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test.toarray() if hasattr(X_test, "toarray") else X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [14]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)


In [16]:
class PlacementNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(PlacementNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim//2, 2)  # 2 classes
        )
    def forward(self, x):
        return self.net(x)

model = PlacementNN(X_train_t.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [17]:
epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Evaluate
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(1)
        acc = accuracy_score(y_test, preds.numpy())

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}")

Epoch 1/30 | Loss: 0.3299 | Val Acc: 0.9065
Epoch 2/30 | Loss: 0.2124 | Val Acc: 0.9085
Epoch 3/30 | Loss: 0.1958 | Val Acc: 0.9125
Epoch 4/30 | Loss: 0.1786 | Val Acc: 0.9250
Epoch 5/30 | Loss: 0.1710 | Val Acc: 0.9285
Epoch 6/30 | Loss: 0.1573 | Val Acc: 0.9305
Epoch 7/30 | Loss: 0.1521 | Val Acc: 0.9295
Epoch 8/30 | Loss: 0.1384 | Val Acc: 0.9330
Epoch 9/30 | Loss: 0.1354 | Val Acc: 0.9350
Epoch 10/30 | Loss: 0.1188 | Val Acc: 0.9375
Epoch 11/30 | Loss: 0.1140 | Val Acc: 0.9415
Epoch 12/30 | Loss: 0.1035 | Val Acc: 0.9450
Epoch 13/30 | Loss: 0.0956 | Val Acc: 0.9480
Epoch 14/30 | Loss: 0.0859 | Val Acc: 0.9525
Epoch 15/30 | Loss: 0.0772 | Val Acc: 0.9515
Epoch 16/30 | Loss: 0.0730 | Val Acc: 0.9545
Epoch 17/30 | Loss: 0.0621 | Val Acc: 0.9555
Epoch 18/30 | Loss: 0.0614 | Val Acc: 0.9590
Epoch 19/30 | Loss: 0.0580 | Val Acc: 0.9560
Epoch 20/30 | Loss: 0.0542 | Val Acc: 0.9580
Epoch 21/30 | Loss: 0.0518 | Val Acc: 0.9550
Epoch 22/30 | Loss: 0.0440 | Val Acc: 0.9535
Epoch 23/30 | Loss:

In [20]:
print("\nClassification Report:")
print(classification_report(y_test, preds.numpy(), target_names=["Not Placed","Placed"]))


Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.97      0.99      0.98      1668
      Placed       0.93      0.85      0.89       332

    accuracy                           0.96      2000
   macro avg       0.95      0.92      0.93      2000
weighted avg       0.96      0.96      0.96      2000

